# FakeNewsNet Title Adaptation

Compare an in-domain TF-IDF baseline with a DistilBERT model adapted from the Kaggle fake-news model. This notebook trains and evaluates on one stratified FakeNewsNet title split. The 20 percent evaluation portion is used during training, so it is validation rather than an untouched final test. Run with the repository root as the working directory.


## Load titles and make one split

The four source CSVs supply their fake/real labels. Both models below use the same split and raw titles.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)

from utils.classical_baselines import baseline_models
from utils.transformer_training import TextClassificationDataset

LABEL_NAMES = ["FAKE", "REAL"]
RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_LENGTH = 128
MODEL_DIR = "models/fakenewsnet_title_model/distilbert_model"
TOKENIZER_DIR = "models/fakenewsnet_title_model/distilbert_tokenizer"
CHECKPOINT_DIR = "models/fakenewsnet_title_model/training"

frames = []
for file_path, label in [
    ("data/FakeNewsNet/gossipcop_fake.csv", "FAKE"),
    ("data/FakeNewsNet/politifact_fake.csv", "FAKE"),
    ("data/FakeNewsNet/gossipcop_real.csv", "REAL"),
    ("data/FakeNewsNet/politifact_real.csv", "REAL"),
]:
    source = pd.read_csv(file_path)
    frames.append(source.assign(label=label, source_file=file_path.split("/")[-1]))

titles = pd.concat(frames, ignore_index=True)
titles["title_text"] = titles["title"].fillna("").astype(str).str.strip()
titles = titles.loc[titles["title_text"].str.len() > 0].reset_index(drop=True)
label_to_id = {name: index for index, name in enumerate(LABEL_NAMES)}
labels = titles["label"].map(label_to_id).astype(int)

train_indices, validation_indices = train_test_split(
    np.arange(len(titles)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)
train_frame = titles.iloc[train_indices].copy()
validation_frame = titles.iloc[validation_indices].copy()
train_labels = labels.iloc[train_indices].tolist()
validation_labels = labels.iloc[validation_indices].tolist()
print("Training rows:", len(train_frame), "Validation rows:", len(validation_frame))


## TF-IDF Linear SVC reference

This model is fitted on FakeNewsNet titles. Its score is an in-domain reference, not Kaggle-to-FakeNewsNet transfer.


In [ ]:
baseline = baseline_models(random_state=RANDOM_STATE)["linear_svc_tfidf"]
baseline.fit(train_frame["title_text"], train_frame["label"])
baseline_predictions = baseline.predict(validation_frame["title_text"])

print(classification_report(
    validation_frame["label"],
    baseline_predictions,
    labels=LABEL_NAMES,
    zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    validation_frame["label"],
    baseline_predictions,
    labels=LABEL_NAMES,
    cmap="Blues",
)
plt.show()


## Adapt DistilBERT and evaluate

Start from the saved Kaggle model. Keep the original two-epoch schedule and best-macro-F1 checkpoint selection. Save the adapted model under models/ for later inspection.


In [ ]:
def compute_training_metrics(eval_pred):
    logits, true_labels = eval_pred
    predicted_labels = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="macro", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(true_labels, predicted_labels)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
    }


tokenizer = DistilBertTokenizerFast.from_pretrained(
    "models/fake_news_model/distilbert_fakenews_tokenizer"
)
model = DistilBertForSequenceClassification.from_pretrained(
    "models/fake_news_model/distilbert_fakenews_model"
)
model.config.id2label = {index: name for index, name in enumerate(LABEL_NAMES)}
model.config.label2id = label_to_id

train_encodings = tokenizer(
    train_frame["title_text"].tolist(),
    truncation=True, padding=True, max_length=MAX_LENGTH,
)
validation_encodings = tokenizer(
    validation_frame["title_text"].tolist(),
    truncation=True, padding=True, max_length=MAX_LENGTH,
)
train_data = TextClassificationDataset(train_encodings, train_labels)
validation_data = TextClassificationDataset(
    validation_encodings, validation_labels
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2.0,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to=[],
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    compute_metrics=compute_training_metrics,
)
trainer.train()
validation_metrics = trainer.evaluate()
display(pd.DataFrame([validation_metrics]))

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print("Saved adapted model to", MODEL_DIR)

logits = trainer.predict(validation_data).predictions
predicted_ids = np.argmax(logits, axis=-1)
predicted_labels = [LABEL_NAMES[index] for index in predicted_ids]
print(classification_report(
    validation_frame["label"],
    predicted_labels,
    labels=LABEL_NAMES,
    zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    validation_frame["label"],
    predicted_labels,
    labels=LABEL_NAMES,
    cmap="Blues",
)
plt.show()


## Interpretation

Compare both in-domain scores with the separate transfer check in notebook 5.0. Adaptation uses FakeNewsNet labels and its validation split, so it cannot be read as transfer without training or as an independent final test.
